In [1]:
from functools import cache
from calitp_data_analysis.gcs_pandas import GCSPandas

@cache
def gcs_pandas():
    return GCSPandas()

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Get NTD ID from warehouse
for all California agences in NTD

In [2]:
# need to get ntd ID of all California agencies
from calitp_data_analysis.sql import query_sql

# also include VOMs for comparison
ca_agencies = query_sql(
    """
    SELECT distinct 
        ntd_id, 
        agency, 
        agency_voms
    FROM cal-itp-data-infra.mart_ntd.dim_annual_service_agencies
    WHERE 
        state = 'CA' 
        AND report_year = 2024
    """,
    as_df=True,
)
ca_agencies = ca_agencies.add_suffix("_mart")
ca_agencies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ntd_id_mart       209 non-null    object 
 1   agency_mart       209 non-null    object 
 2   agency_voms_mart  209 non-null    float64
dtypes: float64(1), object(2)
memory usage: 5.0+ KB


# Read in NTD 2024 Revenue Vehicle Inventory
this list all the revenue vehicles reported by each agency

## NTD Glossary Definitions
- `Active Vehicles`
> The vehicles available to operate in revenue service at the end of the fiscal year, including: 
•   Spares
•   Vehicles temporarily out of service for routine maintenance and minor repairs
•   Operational vehicles

- Active Vehicles in Fleet
>The vehicles in a particular fleet at year-end that are available to operate in revenue service, including: 
•   Spares
•   Vehicles temporarily out of service for routine maintenance and minor repairs

- Number of Active Vehicles in Fleet
>The total number of operational revenue vehicles in the fleet available for general public transit service, including spare or back up revenue vehicles. The total should also include any operational revenue vehicles used by contractors in general public transit service. Non-revenue service vehicles and personal vehicles should not be included. Can be found in: A-30

- `Vehicles in Total Fleet`
>All revenue vehicles held at the end of the fiscal year, including those: 
•   In service;
•   In storage;
•   Emergency contingency; and
•   Awaiting sale.
Can be found in: A-30

- `Vehicles Operated in Annual Maximum Service (VOMS)`
>The number of revenue vehicles operated to meet the annual maximum service requirement. This is the revenue vehicle count during the peak season of the year; on the week and day that maximum service is provided. Vehicles operated in maximum service (VOMS) exclude: 
•   Atypical days; or
•   One-time special events.
Can be found in: B-30, A-30, S-10, Declarations, MR-20

In [3]:
# saved excel to GCS, so use gcs_pandas()
this_gcs_path = "gs://calitp-analytics-data/data-analyses/ntd/"
rev_veh_inv_link = f"{this_gcs_path}2024 Revenue Vehicle Inventory_250820.xlsx"

rev_veh = gcs_pandas().read_excel(rev_veh_inv_link)

rev_veh.columns = rev_veh.columns.str.lower().str.strip().str.replace(" ","_")
rev_veh["ntd_id"] = rev_veh["ntd_id"].astype("str")
rev_veh.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35218 entries, 0 to 35217
Data columns (total 41 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   state/parent_ntd_id                           13899 non-null  object 
 1   ntd_id                                        35218 non-null  object 
 2   agency_name                                   35218 non-null  object 
 3   reporter_type                                 35218 non-null  object 
 4   reporting_module                              35218 non-null  object 
 5   group_plan_sponsor_ntdid                      18739 non-null  object 
 6   group_plan_sponsor_name                       18739 non-null  object 
 7   modes                                         35218 non-null  object 
 8   revenue_vehicle_inventory_id                  35218 non-null  int64  
 9   agency_fleet_id                               35218 non-null 

In [4]:
rev_veh.head(3)

,state/parent_ntd_id,ntd_id,agency_name,reporter_type,reporting_module,group_plan_sponsor_ntdid,group_plan_sponsor_name,modes,revenue_vehicle_inventory_id,agency_fleet_id,modetos_vehicles_operated_in_maximum_service,total_fleet_vehicles,dedicated_fleet,vehicle_type,ownership_type,funding_source,manufacture_year,rebuild_year,type_of_last_renewal,useful_life_benchmark,manufacturer,other_manufacturer_description,model,active_fleet_vehicles,ada_fleet_vehicles,emergency_contingency_vehicles,fuel_type,other_fuel_description,vehicle_length,seating_capacity,standing_capacity,total_miles_on_active_vehicles_during_period,average_lifetime_miles_per_active_vehicles,no_capital_replacement_flag,separate_asset_flag,event_data_recorders,emergency_lighting_system_design,emergency_signage,emergency_path_marking,automated_vehicles_flag,notes
0,NaN,1,King County,Full Reporter,Urban,NaN,NaN,DR/PT,37760,,366.0,2,Y,Cutaway,Owned outright by public agency (OOPA),Non-Federal Public Funds (NFPA),2009.0,NaN,NaN,10.0,Startrans (Supreme Corporation),NaN,SupremeSenator,0,0.0,0.0,Diesel Fuel,NaN,22.0,13,0.0,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,1,King County,Full Reporter,Urban,NaN,NaN,DR/PT,42645,,366.0,14,Y,Cutaway,Owned outright by public agency (OOPA),Non-Federal Public Funds (NFPA),2010.0,NaN,NaN,10.0,Startrans (Supreme Corporation),NaN,SupremeSenator,14,14.0,0.0,Diesel Fuel,NaN,22.0,13,0.0,66135.0,302864.0,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,1,King County,Full Reporter,Urban,NaN,NaN,DR/PT,48057,,366.0,8,Y,Cutaway,Owned outright by public agency (OOPA),Non-Federal Public Funds (NFPA),2011.0,NaN,NaN,10.0,Startrans (Supreme Corporation),NaN,SupremeSenator,8,8.0,0.0,Gasoline,NaN,22.0,13,0.0,80753.0,220649.0,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# filter the NTD list by the NTD id from the warehouse

ca_rev_veh = rev_veh[rev_veh["ntd_id"].isin(
    ca_agencies["ntd_id_mart"].unique().tolist()
)]

In [7]:
# group by / aggregate by active fleet, total fleet, VOMS. 
# maybe there is a difference between all of the. I expect the toal fleet to be the highest

ca_rev_veh = ca_rev_veh.groupby(
    [
        "ntd_id",
        "agency_name",
    ]
).agg({
    "total_fleet_vehicles": "sum",
    "active_fleet_vehicles":"sum",
    "modetos_vehicles_operated_in_maximum_service":"max"
}).reset_index()

In [8]:
ca_rev_veh.head()

,ntd_id,agency_name,total_fleet_vehicles,active_fleet_vehicles,modetos_vehicles_operated_in_maximum_service
0,90003,San Francisco Bay Area Rapid Transit District,785,776,566.0
1,90004,Golden Empire Transit District,160,160,51.0
2,90006,Santa Cruz Metropolitan Transit District,147,147,58.0
3,90008,City of Santa Monica,239,239,124.0
4,90009,San Mateo County Transit District,468,468,178.0


## difference between VOMs and Rev Vehicle Inv.

In [9]:
diff_voms_ref_veh = ca_rev_veh.merge(
    ca_agencies,
    left_on="ntd_id",
    right_on="ntd_id_mart",
    how="inner",
    # suffixes = ("_ntd_excel","_warehouse"),
    # indicator=True # dont really need this since how=inner
)

In [10]:
diff_voms_ref_veh.head()

,ntd_id,agency_name,total_fleet_vehicles,active_fleet_vehicles,modetos_vehicles_operated_in_maximum_service,ntd_id_mart,agency_mart,agency_voms_mart
0,90003,San Francisco Bay Area Rapid Transit District,785,776,566.0,90003,"San Francisco Bay Area Rapid Transit District,...",582.0
1,90004,Golden Empire Transit District,160,160,51.0,90004,Golden Empire Transit District,93.0
2,90006,Santa Cruz Metropolitan Transit District,147,147,58.0,90006,Santa Cruz Metropolitan Transit District,97.0
3,90008,City of Santa Monica,239,239,124.0,90008,"City of Santa Monica, dba: Big Blue Bus",162.0
4,90009,San Mateo County Transit District,468,468,178.0,90009,"San Mateo County Transit District, dba: SamTrans",361.0


## Calculate the differencs in VOMS values between the revenue vehicle inventory and the warehouse
Note, the warehouse table pull from a NTD excel file as well, so VOMS should be the same...

In [11]:
diff_voms_ref_veh["voms_diff"] = diff_voms_ref_veh["agency_voms_mart"] - diff_voms_ref_veh["modetos_vehicles_operated_in_maximum_service"] 

In [12]:
diff_voms_ref_veh["total_fleet_veh_voms_diff"] = diff_voms_ref_veh["total_fleet_vehicles"] - diff_voms_ref_veh["modetos_vehicles_operated_in_maximum_service"]

In [17]:
diff_voms_ref_veh["ntd_id"].nunique()

209

## save data to GCS

In [14]:

diff_voms_ref_veh.to_csv(f"{this_gcs_path}1943_total_revenue_vehicles.csv")

# Count of routes